# Credit Risk Demo - Train LightGBM Model

This notebook trains a LightGBM classifier using **all available features**.

**Algorithm Strengths:** LightGBM provides native categorical feature handling and efficient leaf-wise tree growth for faster training.

## Steps
1. Load preprocessed data (all features)
2. Train LightGBM model
3. Evaluate performance
4. Save model as `lightgbm-model.bst`
5. Upload to MinIO S3

In [ ]:
# Install required packages
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn minio

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from minio import Minio
from urllib.parse import urlparse
import os

print("✓ Libraries imported successfully")

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data from data preparation notebook
X_train = np.load('data/processed/X_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_train = np.load('data/processed/y_train.npy')
y_test = np.load('data/processed/y_test.npy')

# Load feature names
with open('data/processed/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nAll features:")
for f in feature_names:
    print(f"  - {f}")
print(f"\nClass distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

## 2. Train LightGBM Model

LightGBM has **native categorical feature support** - it can handle categorical variables directly without one-hot encoding, which is more efficient and often more accurate.

**Why LightGBM?**
- Native handling of categorical features (loan_intent, home_ownership, default_on_file)
- Leaf-wise tree growth (vs level-wise) finds better splits faster
- Histogram-based algorithm for efficient training
- Different tree-building strategy than XGBoost provides ensemble diversity

LightGBM will analyze the same data as XGBoost but make different predictions due to its unique categorical handling and leaf-wise growth approach.

In [ ]:
# Create LightGBM Dataset
# LightGBM can handle categorical features natively (more efficient than label encoding)
# Specify which features are categorical by their column names
categorical_features = ['loan_intent', 'person_home_ownership', 'cb_person_default_on_file', 'loan_grade']

train_data = lgb.Dataset(
    X_train, 
    label=y_train, 
    feature_name=feature_names,
    categorical_feature=categorical_features
)
test_data = lgb.Dataset(
    X_test, 
    label=y_test, 
    feature_name=feature_names, 
    categorical_feature=categorical_features,
    reference=train_data
)

print("✓ LightGBM Dataset created")
print(f"Total features: {len(feature_names)}")
print(f"Categorical features (native handling): {categorical_features}")
print(f"Numeric features: {len(feature_names) - len(categorical_features)}")

In [ ]:
# Set LightGBM parameters
# Handle class imbalance: Calculate scale_pos_weight (ratio of negative to positive class)
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)  # ~3.58 for this dataset

params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    
    # Boosting configuration
    'boosting_type': 'gbdt',      # Gradient Boosting Decision Tree
    
    # Tree structure - controls model complexity
    'num_leaves': 31,             # Max number of leaves per tree
    'max_depth': -1,              # No limit (controlled by num_leaves instead)
    'min_data_in_leaf': 20,       # Minimum samples per leaf (prevents overfitting)
    
    # Learning parameters
    'learning_rate': 0.05,        # Step size (lower = more conservative, needs more trees)
    
    # Sampling parameters - help prevent overfitting
    'feature_fraction': 0.8,      # Column sampling per tree
    'bagging_fraction': 0.8,      # Row sampling per tree
    'bagging_freq': 5,            # Bagging frequency
    
    # Class imbalance handling
    'scale_pos_weight': scale_pos_weight,  # Balances positive/negative weights
    
    # Other
    'random_state': 42,
    'verbose': -1
}

print("Model parameters:")
for key, value in params.items():
    if key == 'scale_pos_weight':
        print(f"  {key}: {value:.2f} (class imbalance adjustment)")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Train model with early stopping
evals_result = {}

print("Training LightGBM model...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=10),
        lgb.record_evaluation(evals_result)
    ]
)

print(f"\n✓ Training complete!")
print(f"Best iteration: {model.best_iteration}")

In [ ]:
# Plot training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# AUC plot
epochs = range(len(evals_result['train']['auc']))
ax1.plot(epochs, evals_result['train']['auc'], label='Train')
ax1.plot(epochs, evals_result['test']['auc'], label='Test')
ax1.axvline(model.best_iteration - 1, color='red', linestyle='--', label='Best iteration')
ax1.set_xlabel('Boosting Round')
ax1.set_ylabel('AUC')
ax1.set_title('LightGBM Training - AUC')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log Loss plot
ax2.plot(epochs, evals_result['train']['binary_logloss'], label='Train')
ax2.plot(epochs, evals_result['test']['binary_logloss'], label='Test')
ax2.axvline(model.best_iteration - 1, color='red', linestyle='--', label='Best iteration')
ax2.set_xlabel('Boosting Round')
ax2.set_ylabel('Log Loss')
ax2.set_title('LightGBM Training - Log Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Evaluate Model Performance

**Note on Classification Threshold:** We use 0.5 as the decision threshold (probability > 0.5 = default). In production, this threshold should be optimized based on business costs (e.g., cost of missed default vs. cost of rejected good customer).

In [ ]:
# Make predictions on both train and test sets
y_pred_proba_train = model.predict(X_train, num_iteration=model.best_iteration)
y_pred_train = (y_pred_proba_train > 0.5).astype(int)

y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate metrics for both sets
train_accuracy = accuracy_score(y_train, y_pred_train)
train_auc = roc_auc_score(y_train, y_pred_proba_train)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("="*60)
print("LIGHTGBM MODEL PERFORMANCE")
print("="*60)
print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  ROC AUC:   {auc:.4f}")
print(f"\nTrain vs Test Comparison (checking for overfitting):")
print(f"  Train Accuracy: {train_accuracy:.4f}")
print(f"  Test Accuracy:  {accuracy:.4f}")
print(f"  Train AUC:      {train_auc:.4f}")
print(f"  Test AUC:       {auc:.4f}")
if train_auc - auc > 0.05:
    print(f"  ⚠ Significant gap suggests overfitting")
else:
    print(f"  ✓ Good generalization")
print("="*60)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('LightGBM - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# Feature importance
importance = model.feature_importance(importance_type='gain')
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(importance_df.head(10))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance (Gain)')
plt.title('LightGBM - Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Save Model

In [ ]:
# Create lightgbm model directory matching MinIO structure
os.makedirs('models/lightgbm', exist_ok=True)

# Save model in .bst format (MLServer compatible)
model_path = 'models/lightgbm/model.bst'
model.save_model(model_path)

print(f"✓ Model saved to: {model_path}")
print(f"  File size: {os.path.getsize(model_path) / 1024:.2f} KB")

## 5. Upload to MinIO

Upload the trained model to MinIO S3 storage for MLServer to access.

In [ ]:
# MinIO configuration from environment variables
MINIO_ENDPOINT = os.getenv('AWS_S3_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('AWS_ACCESS_KEY_ID')
MINIO_SECRET_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')
MINIO_BUCKET = os.getenv('AWS_S3_BUCKET')

# Parse endpoint to extract protocol and hostname
MINIO_SECURE = False
if MINIO_ENDPOINT:
    parsed = urlparse(MINIO_ENDPOINT if '//' in MINIO_ENDPOINT else f'//{MINIO_ENDPOINT}')
    MINIO_SECURE = parsed.scheme == 'https'
    MINIO_ENDPOINT = parsed.netloc

# Validate required environment variables
required_vars = {
    'AWS_S3_ENDPOINT': MINIO_ENDPOINT,
    'AWS_ACCESS_KEY_ID': MINIO_ACCESS_KEY,
    'AWS_SECRET_ACCESS_KEY': MINIO_SECRET_KEY,
    'AWS_S3_BUCKET': MINIO_BUCKET
}

missing_vars = [var for var, value in required_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

print("MinIO Configuration:")
print(f"  Endpoint: {MINIO_ENDPOINT}")
print(f"  Bucket: {MINIO_BUCKET}")
print(f"  Secure: {MINIO_SECURE}")

In [ ]:
# Initialize MinIO client
try:
    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_SECURE
    )
    print("✓ MinIO client initialized")
except Exception as e:
    print(f"✗ Error initializing MinIO client: {e}")
    raise

In [ ]:
# Check if bucket exists
try:
    if not client.bucket_exists(MINIO_BUCKET):
        client.make_bucket(MINIO_BUCKET)
        print(f"✓ Created bucket: {MINIO_BUCKET}")
    else:
        print(f"✓ Bucket exists: {MINIO_BUCKET}")
except Exception as e:
    print(f"✗ Error checking/creating bucket: {e}")
    raise

In [ ]:
# Upload model to MinIO
try:
    print(f"Uploading lightgbm model to {MINIO_BUCKET}/lightgbm/...")
    
    client.fput_object(MINIO_BUCKET, 'lightgbm/model.bst', 'models/lightgbm/model.bst')
    
    print(f"✓ Model uploaded successfully!")
    print(f"  Location: s3://{MINIO_BUCKET}/lightgbm/model.bst")
    
except Exception as e:
    print(f"✗ Error uploading model: {e}")
    raise

In [ ]:
# Verify upload
print("\nVerifying upload...")
try:
    stat = client.stat_object(MINIO_BUCKET, 'lightgbm/model.bst')
    print(f"✓ Model verified in MinIO:")
    print(f"  Size: {stat.size / 1024:.2f} KB")
    print(f"  Modified: {stat.last_modified}")
except Exception as e:
    print(f"✗ Error verifying upload: {e}")

## Summary

LightGBM model training complete!

### Model Performance
- Check the metrics printed above

### Model Storage
- **Local**: `models/lightgbm-model.bst`
- **MinIO**: `s3://models/lightgbm/model.bst`

### Next Steps
1. Train Sklearn model: `03_train_sklearn.ipynb`
2. Deploy models using MLServer on OpenShift AI

In [ ]:
# Final summary - calculate log loss from predictions
test_logloss = log_loss(y_test, y_pred_proba)

print("="*60)
print("LIGHTGBM MODEL TRAINING SUMMARY")
print("="*60)
print(f"Training: Stopped at iteration {model.best_iteration} (early stopping)")
print(f"\nTest Set Performance:")
print(f"  ROC AUC:   {auc:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Log Loss:  {test_logloss:.4f}")
print(f"\nModel Storage:")
print(f"  Local: models/lightgbm/")
print(f"  MinIO: s3://{MINIO_BUCKET}/lightgbm/")
print("="*60)
print("✓ Ready for deployment!")